# 05 — Baseline Flight Disruption Models

This notebook trains the first predictive baselines for Flight Disruption AI using the leakage-controlled feature table created in Notebook 04.

## Evaluation design
We use a strict temporal split rather than a random split:
- Train: 2020–2023
- Validation: 2024
- Test: 2025

Notebook 04 already saves these assignments in the `split` column. This notebook uses that saved split directly and falls back to `year` only if needed.

The primary task is multiclass classification of `normal`, `delay`, `severe_delay`, and `cancelled`. We report macro-F1 and per-class precision/recall because overall accuracy can hide poor performance on rare disruptions.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    balanced_accuracy_score, f1_score, log_loss
)

cwd = Path.cwd().resolve()
if (cwd / 'data').exists():
    ROOT = cwd
elif (cwd.parent / 'data').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(f'Cannot locate project root from {cwd}')

FEATURE_FILE = ROOT / 'data/processed/ogg_model_features_v1.csv.gz'
SCHEMA_FILE = ROOT / 'data/processed/ogg_model_features_v1_schema.json'

print('Feature file exists:', FEATURE_FILE.exists())
print('Schema exists:', SCHEMA_FILE.exists())


## 1. Load the training table and schema


In [ ]:
df = pd.read_csv(FEATURE_FILE, low_memory=False)
with open(SCHEMA_FILE, 'r') as f:
    schema = json.load(f)

print('Shape:', df.shape)
print('Columns:', len(df.columns))
display(df.head())
display(df['disruption_class'].value_counts(dropna=False).to_frame('count'))


## 2. Restrict to the four primary outcome classes


In [ ]:
target_classes = ['normal', 'delay', 'severe_delay', 'cancelled']
model_df = df[df['disruption_class'].isin(target_classes)].copy()
print('Rows retained:', len(model_df))
display((model_df['disruption_class'].value_counts(normalize=True) * 100).round(3).to_frame('percent'))


## 3. Define temporal train / validation / test partitions

Use the saved split from Notebook 04. No random shuffling is used.


In [ ]:
if 'split' in model_df.columns:
    train_df = model_df[model_df['split'].eq('train')].copy()
    val_df = model_df[model_df['split'].eq('validation')].copy()
    test_df = model_df[model_df['split'].eq('test')].copy()
elif 'year' in model_df.columns:
    train_df = model_df[model_df['year'].between(2020, 2023)].copy()
    val_df = model_df[model_df['year'].eq(2024)].copy()
    test_df = model_df[model_df['year'].eq(2025)].copy()
else:
    raise KeyError(
        "Neither 'split' nor 'year' exists in the training table. Re-run Notebook 04."
    )

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(train_df), len(val_df), len(test_df)],
})
display(split_summary)

for name, part in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print(f'\n{name}')
    display((part['disruption_class'].value_counts(normalize=True) * 100).round(3).to_frame('percent'))


## 4. Select leakage-safe predictors


In [ ]:
exclude = {
    'disruption_class', 'split', 'FlightDate', 'ogg_sched_dt', 'weather_dt',
    'Cancelled', 'CancellationCode', 'Diverted',
    'DepTime', 'ArrTime', 'DepDelay', 'ArrDelay',
    'DepDelayMinutes', 'ArrDelayMinutes',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
}
feature_cols = [c for c in model_df.columns if c not in exclude]
feature_cols = [c for c in feature_cols if not train_df[c].isna().all()]
categorical_cols = [c for c in feature_cols if str(train_df[c].dtype) in ('object', 'category', 'bool')]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print('Features:', len(feature_cols))
print('Numeric:', len(numeric_cols))
print('Categorical:', len(categorical_cols))
display(pd.DataFrame({'feature': feature_cols}))


In [ ]:
X_train, y_train = train_df[feature_cols], train_df['disruption_class']
X_val, y_val = val_df[feature_cols], val_df['disruption_class']
X_test, y_test = test_df[feature_cols], test_df['disruption_class']


## 5. Shared preprocessing


In [ ]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocess = ColumnTransformer([
    ('num', numeric_pipe, numeric_cols),
    ('cat', categorical_pipe, categorical_cols),
])


## 6. Majority-class baseline


In [ ]:
majority_class = y_train.value_counts().idxmax()
majority_pred = np.repeat(majority_class, len(y_val))
print('Majority class:', majority_class)
print('Validation accuracy:', round(accuracy_score(y_val, majority_pred), 4))
print('Validation balanced accuracy:', round(balanced_accuracy_score(y_val, majority_pred), 4))
print('Validation macro-F1:', round(f1_score(y_val, majority_pred, average='macro'), 4))


## 7. Multinomial Logistic Regression baseline


In [ ]:
logreg = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs')),
])
logreg.fit(X_train, y_train)
val_pred_lr = logreg.predict(X_val)
val_prob_lr = logreg.predict_proba(X_val)
print('Validation accuracy:', round(accuracy_score(y_val, val_pred_lr), 4))
print('Validation balanced accuracy:', round(balanced_accuracy_score(y_val, val_pred_lr), 4))
print('Validation macro-F1:', round(f1_score(y_val, val_pred_lr, average='macro'), 4))
print('Validation log loss:', round(log_loss(y_val, val_prob_lr, labels=logreg.classes_), 4))
print(classification_report(y_val, val_pred_lr, labels=target_classes, zero_division=0))


## 8. Random Forest baseline


In [ ]:
rf_preprocess = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical_cols),
])
rf = Pipeline([
    ('preprocess', rf_preprocess),
    ('model', RandomForestClassifier(
        n_estimators=300, max_depth=18, min_samples_leaf=3,
        class_weight='balanced_subsample', n_jobs=-1, random_state=42
    )),
])
rf.fit(X_train, y_train)
val_pred_rf = rf.predict(X_val)
val_prob_rf = rf.predict_proba(X_val)
print('Validation accuracy:', round(accuracy_score(y_val, val_pred_rf), 4))
print('Validation balanced accuracy:', round(balanced_accuracy_score(y_val, val_pred_rf), 4))
print('Validation macro-F1:', round(f1_score(y_val, val_pred_rf, average='macro'), 4))
print('Validation log loss:', round(log_loss(y_val, val_prob_rf, labels=rf.classes_), 4))
print(classification_report(y_val, val_pred_rf, labels=target_classes, zero_division=0))


## 9. Compare validation performance


In [ ]:
results = []
for name, pred in [('majority', majority_pred), ('logistic_regression', val_pred_lr), ('random_forest', val_pred_rf)]:
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_val, pred),
        'balanced_accuracy': balanced_accuracy_score(y_val, pred),
        'macro_f1': f1_score(y_val, pred, average='macro'),
        'weighted_f1': f1_score(y_val, pred, average='weighted'),
    })
results_df = pd.DataFrame(results).sort_values('macro_f1', ascending=False)
display(results_df.round(4))


## 10. Select the better learned baseline and evaluate once on 2025


In [ ]:
lr_f1 = f1_score(y_val, val_pred_lr, average='macro')
rf_f1 = f1_score(y_val, val_pred_rf, average='macro')
best_name, best_model = ('logistic_regression', logreg) if lr_f1 >= rf_f1 else ('random_forest', rf)
print('Selected model:', best_name)
test_pred = best_model.predict(X_test)
test_prob = best_model.predict_proba(X_test)
print('Test accuracy:', round(accuracy_score(y_test, test_pred), 4))
print('Test balanced accuracy:', round(balanced_accuracy_score(y_test, test_pred), 4))
print('Test macro-F1:', round(f1_score(y_test, test_pred, average='macro'), 4))
print('Test weighted-F1:', round(f1_score(y_test, test_pred, average='weighted'), 4))
print('Test log loss:', round(log_loss(y_test, test_prob, labels=best_model.classes_), 4))
print(classification_report(y_test, test_pred, labels=target_classes, zero_division=0))


## 11. Confusion matrix


In [ ]:
cm = confusion_matrix(y_test, test_pred, labels=target_classes)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)
ax.set_xticks(range(len(target_classes)), target_classes, rotation=45, ha='right')
ax.set_yticks(range(len(target_classes)), target_classes)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'2025 Test Confusion Matrix — {best_name}')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center')
plt.tight_layout()
plt.show()


## What to record after running

Please keep the split row counts, class percentages, validation comparison table, and full 2025 classification report. These will determine whether we move next to XGBoost/calibration or first revise the disruption target and imbalance strategy.
